In [37]:
import torch
from transformers import AutoModel, AutoTokenizer
import torch.nn as nn
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from hasoc_model import *
%load_ext autoreload
%autoreload 2
    

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
df_clara = pd.read_csv("hasoc_dataset/train.tsv", sep="\t")
df_clara.columns = ["id", "text", "label_A", "label_B", "label_C"]
df_clara = df_clara[["text", "label_A", "label_B", "label_C"]] 
df_clara = encode_labels(df_clara)

In [39]:
df_claraA = df_clara.dropna(subset=["label_A_enc"])
labelsA_list = df_claraA["label_A_enc"].tolist()
df_claraA = df_claraA["text"].tolist()

df_claraB = df[df["label_A"] == "HOF"].dropna(subset=["label_B_enc"])
labelsB_list = df_claraB["label_B_enc"].tolist()
df_claraB = df_claraB["text"].tolist()

df_claraC = df[(df["label_A"] == "HOF") & (df["label_C"].isin(["UNT", "TIN"]))].dropna(subset=["label_C_enc"])
labelsC_list = df_claraC["label_C_enc"].tolist()
df_claraC = df_claraC["text"].tolist()

In [40]:
len(df_claraA)
n = len(df_claraA)
df_claraA = df_claraA[0:n]
labelsA_list = labelsA_list[0:n]

In [41]:
sentences = [
    "I love all people, no matter where they come from.",
    "That group is disgusting and should not exist.",
    "We should build a more inclusive and respectful community.",
    "You don't belong here. Go back to your country."
]

In [42]:
class Paola(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased", num_outputs=8, bin_outputs=5):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_outputs)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, bin_outputs),
            nn.Sigmoid()
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.regressor(pooled), self.classifier(pooled)

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_paola = Paola().to(device)
model_paola.load_state_dict(torch.load("model2_loaded.pth", map_location=device, weights_only=True))

print("model2_loaded.pth loaded and ready to use!")

model2_loaded.pth loaded and ready to use!


In [44]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
encodings = tokenizer(df_claraA, truncation=True, padding=True, max_length=128, return_tensors="pt")

model_paola.eval()
input_ids_paola = encodings['input_ids'].to(device)
attention_mask_paola = encodings['attention_mask'].to(device)

with torch.no_grad():
    preds_num, preds_bin = model_paola(input_ids=input_ids_paola, attention_mask=attention_mask_paola)

preds_num = preds_num.cpu().numpy()
preds_bin = preds_bin.cpu().numpy()
preds_bin = (preds_bin > 0.5).astype(int)

for idx, sentence in enumerate(df_claraA[0:5]):
    print(f"Sentence: {sentence}")
    print(f"Numerical predictions: {preds_num[idx]}")
    print(f"Binary predictions: {preds_bin[idx]}")
    print()

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.14 GiB. GPU 0 has a total capacity of 19.50 GiB of which 69.88 MiB is free. Process 1916148 has 19.41 GiB memory in use. Of the allocated memory 18.55 GiB is allocated by PyTorch, and 790.12 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

numerical_cols = ['sentiment', 'respect', 'insult', 'humiliate', 'status',
                  'dehumanize', 'attack_defend', 'hatespeech']
                  
binary_cols = ['target_race', 'target_religion', 'target_origin', 'target_gender',
               'target_sexuality']

In [49]:
class Coline(nn.Module):
    def __init__(self, task, model_name=None, num_labels=None, class_weights=None):
        super().__init__()
        self.task = task
        self.model_name = model_name or MODEL_NAMES[task]
        self.num_labels = num_labels or NUM_LABELS[task]
        self.class_weights = class_weights

        self.transformer = AutoModel.from_pretrained(self.model_name)
        hidden_size = self.transformer.config.hidden_size  # usually 768

        self.extra_feat_size = 13  # 8 numerical + 5 binary

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + self.extra_feat_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, self.num_labels)
        )

    def freeze_transformer(self):
        for param in self.transformer.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask, extra_features, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # CLS token
        
        # Concatenate CLS embedding with extra features
        combined = torch.cat((pooled_output, extra_features), dim=1)

        logits = self.classifier(combined)

        if labels is not None:
            return {"logits": logits, "labels": labels}
        return logits


In [50]:
MODEL_NAMES = {
    "A": "roberta-base",
    "B": "GroNLP/hateBERT",
    "C": "GroNLP/hateBERT"
}
NUM_LABELS = {"A": 2, "B": 3, "C": 2}

In [51]:
task = "A"

In [52]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAMES[task], use_fast=True)
encodings = tokenizer(df_claraA, truncation=True, padding=True, return_tensors="pt")
input_ids_clara = encodings['input_ids'].to(device)
attention_mask_clara = encodings['attention_mask'].to(device)

In [53]:
class_weights = compute_class_weights(labelsA_list, NUM_LABELS[task], task=task)

model_colineA = Coline(task="A", model_name="roberta-base", class_weights=class_weights).to(device)
state_dict = torch.load("best_model_A_roberta-base.pth", map_location=device, weights_only=True)

# Strip "roberta." from the beginning of keys that belong to the transformer
transformer_state_dict = {
    k.replace("roberta.", ""): v
    for k, v in state_dict.items()
    if k.startswith("roberta.")
}

# Load into the RobertaModel (your transformer's structure)
model_colineA.transformer.load_state_dict(transformer_state_dict, strict=False)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


_IncompatibleKeys(missing_keys=['pooler.dense.weight', 'pooler.dense.bias'], unexpected_keys=[])

In [54]:
extra_features = np.concatenate([preds_num, preds_bin], axis=1)
extra_features_tensor = torch.tensor(extra_features, dtype=torch.float32)

NameError: name 'preds_num' is not defined

In [ ]:
dataset = Dataset.from_dict({
        "input_ids": input_ids_clara,
        "attention_mask": attention_mask_clara,
        "labels": torch.tensor(labelsA_list, dtype=torch.long).tolist(),
        "extra_features": extra_features_tensor.tolist()
    })

dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_model(task, model_colineA, dataset, tokenizer, resume=True)

In [ ]:
# ------- TASK B ------

In [ ]:
task = "B"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAMES[task], use_fast=True)
encodings = tokenizer(df_claraB, truncation=True, padding=True, return_tensors="pt")
input_ids_clara = encodings['input_ids'].to(device)
attention_mask_clara = encodings['attention_mask'].to(device)

In [ ]:
class_weights = compute_class_weights(labelsB_list, NUM_LABELS[task], task=task)

model_colineB = Coline(task="B", model_name="GroNLP/hateBERT-base", class_weights=class_weights).to(device)
state_dict = torch.load("best_model_B_GroNLP/hateBERT-base.pth", map_location=device, weights_only=True)

transformer_state_dict = {k.replace("transformer.", ""): v for k, v in state_dict.items() if k.startswith("transformer.")}

# Load into the RobertaModel (your transformer's structure)
model_colineB.transformer.load_state_dict(transformer_state_dict, strict=False)

In [ ]:
extra_features = np.concatenate([preds_num, preds_bin], axis=1)
extra_features_tensor = torch.tensor(extra_features, dtype=torch.float32)

In [ ]:
dataset = Dataset.from_dict({
        "input_ids": input_ids_clara,
        "attention_mask": attention_mask_clara,
        "labels": torch.tensor(labelsB_list, dtype=torch.long).tolist(),
        "extra_features": extra_features_tensor.tolist()
    })

dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_model(task, model_colineB, dataset, tokenizer, resume=True)

In [ ]:
# ------- TASK C ------

In [ ]:
task = "C"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAMES[task], use_fast=True)
encodings = tokenizer(df_claraC, truncation=True, padding=True, return_tensors="pt")
input_ids_clara = encodings['input_ids'].to(device)
attention_mask_clara = encodings['attention_mask'].to(device)

In [ ]:
class_weights = compute_class_weights(labelsC_list, NUM_LABELS[task], task=task)

model_colineC = Coline(task="C", model_name="GroNLP/hateBERT-base", class_weights=class_weights).to(device)
state_dict = torch.load("best_model_C_GroNLP/hateBERT-base.pth", map_location=device, weights_only=True)

transformer_state_dict = {k.replace("transformer.", ""): v for k, v in state_dict.items() if k.startswith("transformer.")}

# Load into the RobertaModel (your transformer's structure)
model_colineC.transformer.load_state_dict(transformer_state_dict, strict=False)

In [ ]:
extra_features = np.concatenate([preds_num, preds_bin], axis=1)
extra_features_tensor = torch.tensor(extra_features, dtype=torch.float32)

In [ ]:
dataset = Dataset.from_dict({
        "input_ids": input_ids_clara,
        "attention_mask": attention_mask_clara,
        "labels": torch.tensor(labelsC_list, dtype=torch.long).tolist(),
        "extra_features": extra_features_tensor.tolist()
    })

dataset = dataset.train_test_split(test_size=0.2, seed=42)

train_model(task, model_colineC, dataset, tokenizer, resume=True)